# Amazon — Random Forest with Learned Embeddings
**Features**: 6 engineered numerical features + userId/productId embeddings from EmbeddingANN
**Reads from**: `data/train_features.csv` + `embeddings/`
**Saves to**: `embeddings_RF/`


## Roadmap
```
STEP 1 — Imports & paths
STEP 2 — Check embedding files
STEP 3 — Load data
STEP 4 — Load embeddings
STEP 5 — Scale numerical + build feature matrix
STEP 6 — Train & evaluate
STEP 7 — Feature importance
STEP 8 — Save results
```

---
## Step 1 — Imports & Paths

In [ ]:
import os, json, time, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics       import mean_squared_error, mean_absolute_error

DATA_DIR = r"C:\\Users\\Shivanshu Sirohi\\OneDrive\\Desktop\\Shivanshu\\White Paper\\Personalization\\amazon\\data"
OUT_DIR  = r"C:\\Users\\Shivanshu Sirohi\\OneDrive\\Desktop\\Shivanshu\\White Paper\\Personalization\\amazon\embeddings_RF"
os.makedirs(OUT_DIR, exist_ok=True)


---
## Step 2 — Check Embedding Files

In [ ]:
required = ['user_embeddings.npy','product_embeddings.npy',
            'user_encoder.json','product_encoder.json','ann_results.json']
for fname in required:
    fpath  = os.path.join(r"C:\\Users\\Shivanshu Sirohi\\OneDrive\\Desktop\\Shivanshu\\White Paper\\Personalization\\amazon\\embeddings", fname)
    status = '✓' if os.path.exists(fpath) else '✗ MISSING — run EmbeddingANN notebook first'
    print(f"  {status}  {fname}")


---
## Step 3 — Load Data

In [ ]:
df_train = pd.read_csv(os.path.join(DATA_DIR, 'train_features.csv'))
df_val   = pd.read_csv(os.path.join(DATA_DIR, 'val_features.csv'))
df_test  = pd.read_csv(os.path.join(DATA_DIR, 'test_features.csv'))
print(f"Train : {df_train.shape}  Val : {df_val.shape}  Test : {df_test.shape}")


In [ ]:
FEATURE_COLS = [
    'user_avg_rating', 'user_rating_count',
    'product_avg_rating', 'product_rating_count', 'product_rating_std',
    'days_since_review'
]
TARGET = 'ratings'


---
## Step 4 — Load Embeddings

In [ ]:
EMB_DIR = r"C:\\Users\\Shivanshu Sirohi\\OneDrive\\Desktop\\Shivanshu\\White Paper\\Personalization\\amazon\\embeddings"

# Load weight matrices
user_weights = np.load(os.path.join(EMB_DIR, 'user_embeddings.npy'))
prod_weights = np.load(os.path.join(EMB_DIR, 'product_embeddings.npy'))

# Load encoders
with open(os.path.join(EMB_DIR, 'user_encoder.json')) as f:
    user2idx = json.load(f)
with open(os.path.join(EMB_DIR, 'product_encoder.json')) as f:
    product2idx = json.load(f)

print(f"User embedding matrix    : {user_weights.shape}")
print(f"Product embedding matrix : {prod_weights.shape}")
print(f"User vocab size          : {len(user2idx):,}")
print(f"Product vocab size       : {len(product2idx):,}")


In [ ]:
def lookup_embeddings(df, user2idx, product2idx, user_weights, prod_weights):
    """
    For each row, look up userId and productId in the embedding tables.
    Returns concatenated embedding vectors — shape (n_rows, user_dim + prod_dim).
    Unseen IDs map to index 0 (<UNK>) which is a zero vector.
    """
    user_idx = df['userId'].map(lambda x: user2idx.get(x, 0)).values
    prod_idx = df['productId'].map(lambda x: product2idx.get(x, 0)).values

    user_vecs = user_weights[user_idx]  # (n_rows, user_emb_dim)
    prod_vecs = prod_weights[prod_idx]  # (n_rows, prod_emb_dim)

    return np.concatenate([user_vecs, prod_vecs], axis=1)


def build_feature_matrix(df_, num_arr, user2idx, product2idx,
                          user_weights, prod_weights):
    """Concatenate scaled numerical features + embedding vectors."""
    emb_part = lookup_embeddings(df_, user2idx, product2idx,
                                 user_weights, prod_weights)
    return np.concatenate([num_arr, emb_part], axis=1)


---
## Step 5 — Scale Numerical + Build Feature Matrix

Final feature matrix = scaled numerical (6) + user embedding (32) + product embedding (32)
= 70 features total per row.


In [ ]:
scaler  = StandardScaler()
X_train_num = scaler.fit_transform(df_train[FEATURE_COLS])
X_val_num   = scaler.transform(df_val[FEATURE_COLS])
X_test_num  = scaler.transform(df_test[FEATURE_COLS])
y_train = df_train[TARGET].values
y_val   = df_val[TARGET].values
y_test  = df_test[TARGET].values
print(f"Numerical feature matrix — train: {X_train_num.shape}")


In [ ]:
X_train = build_feature_matrix(df_train, X_train_num,
                                user2idx, product2idx, user_weights, prod_weights)
X_val   = build_feature_matrix(df_val,   X_val_num,
                                user2idx, product2idx, user_weights, prod_weights)
X_test  = build_feature_matrix(df_test,  X_test_num,
                                user2idx, product2idx, user_weights, prod_weights)

user_emb_dim = user_weights.shape[1]
prod_emb_dim = prod_weights.shape[1]

print(f"Feature matrix — train: {X_train.shape}")
print(f"  {len(FEATURE_COLS)} numerical  +  {user_emb_dim} user emb dims  +  {prod_emb_dim} prod emb dims")


---
## Step 6 — Train & Evaluate
Default Random Forest hyperparameters.

In [ ]:
def get_metrics(y_true, y_pred):
    rmse = round(float(np.sqrt(mean_squared_error(y_true, y_pred))), 4)
    mae  = round(float(mean_absolute_error(y_true, y_pred)), 4)
    return rmse, mae


In [ ]:
t0    = time.perf_counter()
model = RandomForestRegressor(random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
train_time = time.perf_counter() - t0

In [ ]:
tr_rmse,  tr_mae  = get_metrics(y_train, model.predict(X_train))
val_rmse, val_mae = get_metrics(y_val,   model.predict(X_val))
te_rmse,  te_mae  = get_metrics(y_test,  model.predict(X_test))

print("=" * 55)
print(f"Random Forest [Embeddings] — Results")
print("=" * 55)
print(f"  Train      RMSE: {tr_rmse}   MAE: {tr_mae}")
print(f"  Validation RMSE: {val_rmse}   MAE: {val_mae}")
print(f"  Test       RMSE: {te_rmse}   MAE: {te_mae}")
print(f"  Train time : {train_time:.2f}s")
print(f"  Train-Test gap : {round(te_rmse - tr_rmse, 4)}")


---
## Step 7 — Feature Importance

In [ ]:
feat_names = (FEATURE_COLS
              + [f'user_emb_{i}' for i in range(user_emb_dim)]
              + [f'prod_emb_{i}' for i in range(prod_emb_dim)])
imp_df = (pd.DataFrame({'feature': feat_names,
                         'importance': model.feature_importances_})
            .sort_values('importance', ascending=False).head(15))
print("Top 15 features by importance:")
print(imp_df.to_string(index=False))

---
## Step 8 — Save Results

In [ ]:
result = {
    'model': 'Random Forest [Embeddings]',
    'train_rmse': tr_rmse, 'val_rmse': val_rmse, 'test_rmse': te_rmse,
    'train_mae':  tr_mae,  'val_mae':  val_mae,  'test_mae':  te_mae,
    'train_time_s': round(train_time, 2),
}
pd.DataFrame([result]).to_csv(os.path.join(r"C:\\Users\\Shivanshu Sirohi\\OneDrive\\Desktop\\Shivanshu\\White Paper\\Personalization\\amazon\\embeddings_RF", 'rf_results.csv'), index=False)
with open(os.path.join(r"C:\\Users\\Shivanshu Sirohi\\OneDrive\\Desktop\\Shivanshu\\White Paper\\Personalization\\amazon\\embeddings_RF", 'rf_result.json'), 'w') as f:
    json.dump(result, f, indent=2)
print(f"Saved → C:\\Users\\Shivanshu Sirohi\\OneDrive\\Desktop\\Shivanshu\\White Paper\\Personalization\\amazon\\embeddings_RF")
print(f"Test RMSE: {te_rmse}  Test MAE: {te_mae}")
